# Bootstrap error bars for already-selected single models — v2

When `test_predictions.csv` is present in both the real-only and the augmented directory, `test.csv` is not needed.

Only when a branch has no `test_predictions.csv` does the notebook look for `cycleN/test.csv` or `cycleN/split/test.csv` and regenerate the predictions with `final_best_model.h5`.


In [1]:
# -*- coding: utf-8 -*-
"""
Bootstrap error bars for the original single final DNN models.

What this script does
---------------------
1. For every cycle, identify:
   - real-only final model: artifacts_dnn_base
   - best augmented final model: the GAN scenario with the lowest Best_Val_Loss
     in scenario_summary.csv, unless manually overridden.
2. Prefer the already-saved test_predictions.csv from the original DNN_GAN run.
   This preserves exactly the point estimates used in the original figure.
3. If test_predictions.csv is missing, regenerate predictions from:
   final_best_model.h5, sx_final.joblib, sy_final.joblib, and split/test.csv.
4. Calculate:
   - single-model test MAE
   - 95% percentile bootstrap interval of each MAE
   - paired-bootstrap interval of
       Delta MAE = MAE_augmented - MAE_real-only
5. Save CSV summaries and a three-panel figure.

Interpretation
--------------
The error bars quantify uncertainty in the test MAE caused by the limited held-out
test samples. They do not quantify random-seed, training-split, GAN, or experimental
measurement uncertainty.
"""

import json
import math
import os
from pathlib import Path
from typing import Dict, Optional, Tuple

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error


# ============================================================
# 1. User configuration
# ============================================================
BASE_DIR = Path.cwd()

# Point directly at the final model directories that were already selected.
# Each directory should hold the following files where they exist.
#   final_best_model.h5
#   sx_final.joblib
#   sy_final.joblib
#   test_predictions.csv
#
# With test_predictions.csv present, the existing Fig. 2 predictions are used as they are.
# Without it, predictions are recomputed from final_best_model.h5 and the scaler.
ARTIFACT_DIRS = {
    (1, "real-only"): BASE_DIR / "cycle1" / "artifacts_dnn_base",
    (1, "augmented"): BASE_DIR / "cycle1" / "artifacts_dnn_augmented",

    (2, "real-only"): BASE_DIR / "cycle2" / "artifacts_dnn_base",
    (2, "augmented"): BASE_DIR / "cycle2" / "artifacts_dnn_augmented",

    (3, "real-only"): BASE_DIR / "cycle3" / "artifacts_dnn_base",
    (3, "augmented"): BASE_DIR / "cycle3" / "artifacts_dnn_augmented",
}

# test.csv is not needed when test_predictions.csv is present.
# It is searched for in the candidate paths below only when test_predictions.csv is missing.
CYCLE_TEST_CANDIDATES = {
    1: [
        BASE_DIR / "cycle1" / "test.csv",
        BASE_DIR / "cycle1" / "split" / "test.csv",
    ],
    2: [
        BASE_DIR / "cycle2" / "test.csv",
        BASE_DIR / "cycle2" / "split" / "test.csv",
    ],
    3: [
        BASE_DIR / "cycle3" / "test.csv",
        BASE_DIR / "cycle3" / "split" / "test.csv",
    ],
}

TEST_PREDICTIONS_FILENAME = "test_predictions.csv"
MODEL_FILENAME = "final_best_model.h5"
X_SCALER_FILENAME = "sx_final.joblib"
Y_SCALER_FILENAME = "sy_final.joblib"

OUTPUT_DIR = BASE_DIR / "artifacts_single_model_bootstrap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_CONFIDENCE = 0.95
BOOTSTRAP_SEED = 42

# Keep this False if the original code did not apply abs() to S_ANE separately.
TAKE_ABS_S_ANE = False

SELECTED_BRANCH = {
    1: "augmented",
    2: "real-only",
    3: "augmented",
}
MARK_SELECTED_BRANCH = True

COMP_COLS = ["Fe", "Co", "Mn", "Ga", "Al", "Si", "Ge", "Pt"]
TARGETS = ["kxx", "S_ANE"]

PREDICTION_COLUMNS = {
    "kxx": ("y_true_kxx", "y_pred_kxx"),
    "S_ANE": ("y_true_S_ANE", "y_pred_S_ANE"),
}


# ============================================================
# 2. Feature engineering used only as fallback
# ============================================================
R_GAS = 8.31446261815324

ELEMENT_PROPS = {
    "Fe": {"radius": 1.26,  "vec": 8.0,  "weight": 55.845,     "en": 1.83},
    "Co": {"radius": 1.25,  "vec": 9.0,  "weight": 58.933195,  "en": 1.88},
    "Mn": {"radius": 1.37,  "vec": 7.0,  "weight": 54.938044,  "en": 1.55},
    "Ga": {"radius": 1.408, "vec": 3.0,  "weight": 69.723,     "en": 1.81},
    "Al": {"radius": 1.429, "vec": 3.0,  "weight": 26.9815385, "en": 1.61},
    "Si": {"radius": 1.316, "vec": 4.0,  "weight": 28.085,     "en": 1.90},
    "Ge": {"radius": 1.366, "vec": 4.0,  "weight": 72.63,      "en": 2.01},
    "Pt": {"radius": 1.39,  "vec": 10.0, "weight": 195.084,    "en": 2.28},
}


def closure(array: np.ndarray, axis: int = -1) -> np.ndarray:
    values = np.asarray(array, dtype=np.float64)
    sums = values.sum(axis=axis, keepdims=True)
    sums[sums == 0] = 1.0
    return values / sums


def multiplicative_replacement(
    array: np.ndarray,
    delta: float = 1e-3,
) -> np.ndarray:
    values = closure(array)
    zero_mask = values == 0
    if not zero_mask.any():
        return values

    replaced = values.copy()
    for row_index in range(values.shape[0]):
        zero_count = int(zero_mask[row_index].sum())
        if zero_count == 0:
            continue

        nonzero_sum = values[row_index, ~zero_mask[row_index]].sum()
        if nonzero_sum == 0:
            replaced[row_index] = 1.0 / values.shape[1]
            continue

        replaced[row_index, zero_mask[row_index]] = delta
        replaced[row_index, ~zero_mask[row_index]] = (
            values[row_index, ~zero_mask[row_index]]
            * ((1.0 - zero_count * delta) / nonzero_sum)
        )

    return closure(replaced)


def helmert_submatrix(dimension: int) -> np.ndarray:
    matrix = np.zeros((dimension, dimension - 1), dtype=np.float64)
    for index in range(1, dimension):
        coefficient = 1.0 / math.sqrt(index * (index + 1))
        matrix[:index, index - 1] = coefficient
        matrix[index, index - 1] = -index * coefficient
    return matrix


def ilr_transform(compositions: np.ndarray) -> np.ndarray:
    values = multiplicative_replacement(compositions, delta=1e-3)
    log_values = np.log(closure(values))
    clr = log_values - log_values.mean(axis=1, keepdims=True)
    return clr @ helmert_submatrix(values.shape[1])


def batch_atomic_properties(compositions: np.ndarray) -> np.ndarray:
    values = closure(np.asarray(compositions, dtype=np.float64))

    radius = np.array(
        [ELEMENT_PROPS[element]["radius"] for element in COMP_COLS],
        dtype=np.float64,
    )
    vec = np.array(
        [ELEMENT_PROPS[element]["vec"] for element in COMP_COLS],
        dtype=np.float64,
    )
    weight = np.array(
        [ELEMENT_PROPS[element]["weight"] for element in COMP_COLS],
        dtype=np.float64,
    )
    electronegativity = np.array(
        [ELEMENT_PROPS[element]["en"] for element in COMP_COLS],
        dtype=np.float64,
    )

    radius_average = values @ radius
    atomic_size_difference = np.sqrt(
        np.sum(
            values
            * (1.0 - radius[None, :] / radius_average[:, None]) ** 2,
            axis=1,
        )
    )

    vec_average = values @ vec
    vec_std = np.sqrt(
        np.sum(values * (vec[None, :] - vec_average[:, None]) ** 2, axis=1)
    )

    weight_average = values @ weight

    values_safe = np.clip(values, 1e-12, None)
    mixing_entropy = -R_GAS * np.sum(
        values_safe * np.log(values_safe),
        axis=1,
    )

    en_average = values @ electronegativity
    en_std = np.sqrt(
        np.sum(
            values
            * (electronegativity[None, :] - en_average[:, None]) ** 2,
            axis=1,
        )
    )

    return np.column_stack(
        [
            radius_average,
            atomic_size_difference,
            vec_average,
            vec_std,
            weight_average,
            mixing_entropy,
            en_average,
            en_std,
        ]
    )


def make_features(compositions: np.ndarray) -> np.ndarray:
    ilr = ilr_transform(compositions)
    calculated = batch_atomic_properties(compositions)
    return np.hstack([ilr, calculated]).astype(np.float32)


# ============================================================
# 3. Scenario and prediction loading
# ============================================================
def validate_prediction_frame(
    predictions: pd.DataFrame,
    source_path: Path,
) -> pd.DataFrame:
    required_columns = [
        "y_true_kxx",
        "y_pred_kxx",
        "y_true_S_ANE",
        "y_pred_S_ANE",
    ]
    missing = [
        column
        for column in required_columns
        if column not in predictions.columns
    ]
    if missing:
        raise KeyError(
            f"{source_path} is missing columns: {missing}"
        )

    output = predictions[required_columns].copy()
    output = output.apply(pd.to_numeric, errors="coerce")
    output = output.dropna().reset_index(drop=True)

    if len(output) == 0:
        raise ValueError(
            f"No valid test predictions remain after dropna: {source_path}"
        )

    if TAKE_ABS_S_ANE:
        output["y_true_S_ANE"] = output["y_true_S_ANE"].abs()
        output["y_pred_S_ANE"] = output["y_pred_S_ANE"].abs()

    return output


def regenerate_predictions_from_model(
    test_path: Path,
    artifact_dir: Path,
) -> pd.DataFrame:
    try:
        import tensorflow as tf
    except ImportError as error:
        raise ImportError(
            "test_predictions.csv is missing and TensorFlow is required "
            "to reload final_best_model.h5."
        ) from error

    model_path = artifact_dir / MODEL_FILENAME
    x_scaler_path = artifact_dir / X_SCALER_FILENAME
    y_scaler_path = artifact_dir / Y_SCALER_FILENAME
    required_paths = [
        model_path,
        x_scaler_path,
        y_scaler_path,
        test_path,
    ]
    missing_paths = [path for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            "Files required to regenerate test predictions are missing:\n"
            + "\n".join(str(path) for path in missing_paths)
        )

    test_data = pd.read_csv(test_path)
    required_columns = COMP_COLS + TARGETS
    missing_columns = [
        column for column in required_columns
        if column not in test_data.columns
    ]
    if missing_columns:
        raise KeyError(
            f"{test_path} is missing columns: {missing_columns}"
        )

    test_data = test_data.dropna(subset=required_columns).reset_index(drop=True)

    x_scaler = joblib.load(x_scaler_path)
    y_scaler = joblib.load(y_scaler_path)
    model = tf.keras.models.load_model(model_path, compile=False)

    compositions = test_data[COMP_COLS].to_numpy(dtype=np.float64)
    x_scaled = x_scaler.transform(make_features(compositions))
    pred_scaled = model(x_scaled, training=False).numpy()
    pred_physical = y_scaler.inverse_transform(pred_scaled)

    predictions = pd.DataFrame(
        {
            "y_true_kxx": test_data["kxx"].to_numpy(dtype=float),
            "y_pred_kxx": pred_physical[:, 0],
            "y_true_S_ANE": test_data["S_ANE"].to_numpy(dtype=float),
            "y_pred_S_ANE": pred_physical[:, 1],
        }
    )

    prediction_path = artifact_dir / TEST_PREDICTIONS_FILENAME
    predictions.to_csv(
        prediction_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Regenerated and saved: {prediction_path}")

    return validate_prediction_frame(predictions, prediction_path)


def load_single_model_predictions(
    test_path: Optional[Path],
    artifact_dir: Path,
) -> pd.DataFrame:
    prediction_path = artifact_dir / TEST_PREDICTIONS_FILENAME

    if prediction_path.exists():
        predictions = pd.read_csv(prediction_path)
        print(f"Loaded existing predictions: {prediction_path}")
        return validate_prediction_frame(predictions, prediction_path)

    if test_path is None:
        raise FileNotFoundError(
            f"{prediction_path} is missing, but no test.csv path was resolved."
        )

    print(
        f"Prediction file not found; loading final model instead: "
        f"{artifact_dir / MODEL_FILENAME}"
    )
    return regenerate_predictions_from_model(
        test_path=test_path,
        artifact_dir=artifact_dir,
    )


# ============================================================
# 4. Bootstrap statistics
# ============================================================
def percentile_bootstrap_mae(
    true_values: np.ndarray,
    predicted_values: np.ndarray,
    iterations: int,
    confidence: float,
    seed: int,
) -> Dict[str, float]:
    true_array = np.asarray(true_values, dtype=float)
    predicted_array = np.asarray(predicted_values, dtype=float)

    if len(true_array) != len(predicted_array):
        raise ValueError("true and predicted arrays have different lengths")
    if len(true_array) == 0:
        raise ValueError("Cannot bootstrap an empty test set")

    mae = float(mean_absolute_error(true_array, predicted_array))

    rng = np.random.default_rng(seed)
    boot_values = np.empty(iterations, dtype=float)
    sample_count = len(true_array)

    for bootstrap_index in range(iterations):
        indices = rng.integers(0, sample_count, size=sample_count)
        boot_values[bootstrap_index] = mean_absolute_error(
            true_array[indices],
            predicted_array[indices],
        )

    alpha = (1.0 - confidence) / 2.0
    low = float(np.quantile(boot_values, alpha))
    high = float(np.quantile(boot_values, 1.0 - alpha))

    return {
        "MAE": mae,
        "MAE_CI_low": low,
        "MAE_CI_high": high,
        "MAE_error_low": mae - low,
        "MAE_error_high": high - mae,
    }


def paired_bootstrap_mae_difference(
    true_values: np.ndarray,
    pred_real: np.ndarray,
    pred_augmented: np.ndarray,
    iterations: int,
    confidence: float,
    seed: int,
) -> Dict[str, float]:
    true_array = np.asarray(true_values, dtype=float)
    real_array = np.asarray(pred_real, dtype=float)
    augmented_array = np.asarray(pred_augmented, dtype=float)

    if not (
        len(true_array)
        == len(real_array)
        == len(augmented_array)
    ):
        raise ValueError(
            "Paired bootstrap arrays have different lengths"
        )

    abs_error_real = np.abs(true_array - real_array)
    abs_error_augmented = np.abs(true_array - augmented_array)

    delta = float(
        abs_error_augmented.mean() - abs_error_real.mean()
    )

    rng = np.random.default_rng(seed)
    boot_delta = np.empty(iterations, dtype=float)
    sample_count = len(true_array)

    for bootstrap_index in range(iterations):
        indices = rng.integers(0, sample_count, size=sample_count)
        boot_delta[bootstrap_index] = (
            abs_error_augmented[indices].mean()
            - abs_error_real[indices].mean()
        )

    alpha = (1.0 - confidence) / 2.0
    low = float(np.quantile(boot_delta, alpha))
    high = float(np.quantile(boot_delta, 1.0 - alpha))

    if high < 0:
        interpretation = "augmented_lower_MAE"
    elif low > 0:
        interpretation = "real_only_lower_MAE"
    else:
        interpretation = "inconclusive_zero_in_interval"

    return {
        "Delta_MAE_aug_minus_real": delta,
        "Delta_CI_low": low,
        "Delta_CI_high": high,
        "Bootstrap_fraction_delta_below_zero": float(
            np.mean(boot_delta < 0)
        ),
        "Interpretation": interpretation,
    }


def check_paired_test_sets(
    real_predictions: pd.DataFrame,
    augmented_predictions: pd.DataFrame,
    cycle: int,
) -> None:
    if len(real_predictions) != len(augmented_predictions):
        raise ValueError(
            f"Cycle {cycle}: real-only and augmented test sets have "
            f"different sizes ({len(real_predictions)} vs "
            f"{len(augmented_predictions)})."
        )

    for target in TARGETS:
        true_column = PREDICTION_COLUMNS[target][0]
        if not np.allclose(
            real_predictions[true_column].to_numpy(dtype=float),
            augmented_predictions[true_column].to_numpy(dtype=float),
            rtol=1e-7,
            atol=1e-9,
            equal_nan=False,
        ):
            raise ValueError(
                f"Cycle {cycle}: true {target} values differ between "
                "real-only and augmented prediction files. Paired "
                "bootstrap requires the same test samples in the same order."
            )


# ============================================================
# 5. Figure
# ============================================================
def plot_mae_with_bootstrap(
    summary: pd.DataFrame,
    output_png: Path,
    output_pdf: Path,
) -> None:
    cycles = sorted(summary["cycle"].unique())
    branches = ["real-only", "augmented"]

    # Use Matplotlib's default color cycle rather than hard-coding colors.
    default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    kxx_color = default_colors[1 % len(default_colors)]
    sane_color = default_colors[0]

    figure, axes = plt.subplots(
        1,
        len(cycles),
        figsize=(5.0 * len(cycles), 5.3),
        squeeze=False,
    )

    for column, cycle in enumerate(cycles):
        axis_kxx = axes[0, column]
        axis_sane = axis_kxx.twinx()

        cycle_data = summary[summary["cycle"] == cycle]
        x_positions = np.array([0.0, 1.0])
        bar_width = 0.28

        kxx_subset = (
            cycle_data[cycle_data["target"] == "kxx"]
            .set_index("branch")
            .reindex(branches)
        )
        sane_subset = (
            cycle_data[cycle_data["target"] == "S_ANE"]
            .set_index("branch")
            .reindex(branches)
        )

        kxx_values = kxx_subset["MAE"].to_numpy(dtype=float)
        kxx_yerr = np.vstack(
            [
                kxx_values
                - kxx_subset["MAE_CI_low"].to_numpy(dtype=float),
                kxx_subset["MAE_CI_high"].to_numpy(dtype=float)
                - kxx_values,
            ]
        )

        sane_values = sane_subset["MAE"].to_numpy(dtype=float)
        sane_yerr = np.vstack(
            [
                sane_values
                - sane_subset["MAE_CI_low"].to_numpy(dtype=float),
                sane_subset["MAE_CI_high"].to_numpy(dtype=float)
                - sane_values,
            ]
        )

        kxx_bars = axis_kxx.bar(
            x_positions - bar_width / 2,
            kxx_values,
            width=bar_width,
            yerr=kxx_yerr,
            capsize=4,
            label=r"$\kappa$ MAE",
            color=kxx_color,
            alpha=0.9,
        )
        sane_bars = axis_sane.bar(
            x_positions + bar_width / 2,
            sane_values,
            width=bar_width,
            yerr=sane_yerr,
            capsize=4,
            label=r"$|S_{\mathrm{ANE}}|$ MAE",
            color=sane_color,
            alpha=0.75,
        )

        # Hatch the real-only bars to reproduce the visual distinction
        # from the original figure.
        kxx_bars[0].set_hatch("////")
        kxx_bars[0].set_facecolor("none")
        kxx_bars[0].set_edgecolor(kxx_color)
        kxx_bars[0].set_linewidth(1.8)

        sane_bars[0].set_hatch("////")
        sane_bars[0].set_facecolor("none")
        sane_bars[0].set_edgecolor(sane_color)
        sane_bars[0].set_linewidth(1.8)

        axis_kxx.set_xticks(x_positions)
        axis_kxx.set_xticklabels(["No Aug.", "Aug."])
        axis_kxx.set_ylabel(
            r"$\kappa$ MAE (W m$^{-1}$ K$^{-1}$)"
        )
        axis_sane.set_ylabel(
            r"$|S_{\mathrm{ANE}}|$ MAE ($\mu$V K$^{-1}$)"
        )
        axis_kxx.set_title(f"Cycle {cycle}")
        axis_kxx.set_ylim(bottom=0)
        axis_sane.set_ylim(bottom=0)
        axis_kxx.grid(axis="y", alpha=0.25)

        if MARK_SELECTED_BRANCH:
            selected = SELECTED_BRANCH.get(cycle)
            if selected in branches:
                selected_index = branches.index(selected)
                axis_kxx.text(
                    x_positions[selected_index],
                    1.02,
                    "Selected",
                    transform=axis_kxx.get_xaxis_transform(),
                    ha="center",
                    va="bottom",
                    fontsize=9,
                    fontweight="bold",
                )

        if column == 0:
            handles_kxx, labels_kxx = axis_kxx.get_legend_handles_labels()
            handles_sane, labels_sane = axis_sane.get_legend_handles_labels()
            axis_kxx.legend(
                handles_kxx + handles_sane,
                labels_kxx + labels_sane,
                loc="upper center",
                bbox_to_anchor=(0.5, 1.20),
                ncol=2,
                frameon=False,
            )

    figure.tight_layout()
    figure.savefig(output_png, dpi=300, bbox_inches="tight")
    figure.savefig(output_pdf, bbox_inches="tight")
    plt.close(figure)


def resolve_test_path(cycle: int, branch_dirs: Dict[str, Path]) -> Optional[Path]:
    """
    Return None when both artifact folders already contain test_predictions.csv.
    Otherwise return the first existing test.csv candidate.
    """
    prediction_files_exist = all(
        (artifact_dir / TEST_PREDICTIONS_FILENAME).exists()
        for artifact_dir in branch_dirs.values()
    )
    if prediction_files_exist:
        return None

    candidates = CYCLE_TEST_CANDIDATES.get(cycle, [])
    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Cycle {cycle}: at least one artifact folder does not contain "
        f"{TEST_PREDICTIONS_FILENAME}, and no test.csv was found.\n"
        f"Searched paths:\n{searched}"
    )


# ============================================================
# 6. Main
# ============================================================
def main() -> None:
    prediction_frames: Dict[Tuple[int, str], pd.DataFrame] = {}
    artifact_records = []
    summary_rows = []
    paired_rows = []

    cycles = sorted({cycle for cycle, _ in ARTIFACT_DIRS.keys()})

    for cycle in cycles:
        branch_dirs = {
            branch: ARTIFACT_DIRS[(cycle, branch)]
            for branch in ["real-only", "augmented"]
        }

        test_path = resolve_test_path(
            cycle=cycle,
            branch_dirs=branch_dirs,
        )

        if test_path is None:
            print(
                f"[Cycle {cycle}] Both branches contain "
                f"{TEST_PREDICTIONS_FILENAME}; test.csv is not required."
            )
        else:
            print(f"[Cycle {cycle}] Fallback test file: {test_path}")

        for branch, artifact_dir in branch_dirs.items():
            if not artifact_dir.exists():
                raise FileNotFoundError(
                    f"Artifact directory does not exist: {artifact_dir}"
                )

            predictions = load_single_model_predictions(
                test_path=test_path,
                artifact_dir=artifact_dir,
            )
            prediction_frames[(cycle, branch)] = predictions

            artifact_records.append(
                {
                    "cycle": cycle,
                    "branch": branch,
                    "artifact_dir": str(artifact_dir.resolve()),
                    "model_path": str(
                        (artifact_dir / MODEL_FILENAME).resolve()
                    ),
                    "prediction_path": str(
                        (
                            artifact_dir
                            / TEST_PREDICTIONS_FILENAME
                        ).resolve()
                    ),
                    "selected_for_active_learning": (
                        SELECTED_BRANCH.get(cycle) == branch
                    ),
                }
            )

            for target_index, target in enumerate(TARGETS):
                true_column, pred_column = PREDICTION_COLUMNS[target]
                bootstrap_result = percentile_bootstrap_mae(
                    true_values=predictions[true_column].to_numpy(
                        dtype=float
                    ),
                    predicted_values=predictions[pred_column].to_numpy(
                        dtype=float
                    ),
                    iterations=BOOTSTRAP_ITERATIONS,
                    confidence=BOOTSTRAP_CONFIDENCE,
                    seed=(
                        BOOTSTRAP_SEED
                        + 1000 * cycle
                        + 100 * target_index
                        + (0 if branch == "real-only" else 1)
                    ),
                )

                summary_rows.append(
                    {
                        "cycle": cycle,
                        "branch": branch,
                        "target": target,
                        "n_test": len(predictions),
                        **bootstrap_result,
                        "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
                        "bootstrap_confidence": (
                            BOOTSTRAP_CONFIDENCE
                        ),
                        "selected_for_active_learning": (
                            SELECTED_BRANCH.get(cycle) == branch
                        ),
                    }
                )

        real_predictions = prediction_frames[(cycle, "real-only")]
        augmented_predictions = prediction_frames[
            (cycle, "augmented")
        ]
        check_paired_test_sets(
            real_predictions=real_predictions,
            augmented_predictions=augmented_predictions,
            cycle=cycle,
        )

        for target_index, target in enumerate(TARGETS):
            true_column, pred_column = PREDICTION_COLUMNS[target]

            paired_result = paired_bootstrap_mae_difference(
                true_values=real_predictions[true_column].to_numpy(
                    dtype=float
                ),
                pred_real=real_predictions[pred_column].to_numpy(
                    dtype=float
                ),
                pred_augmented=augmented_predictions[
                    pred_column
                ].to_numpy(dtype=float),
                iterations=BOOTSTRAP_ITERATIONS,
                confidence=BOOTSTRAP_CONFIDENCE,
                seed=(
                    BOOTSTRAP_SEED
                    + 10000
                    + 1000 * cycle
                    + target_index
                ),
            )

            paired_rows.append(
                {
                    "cycle": cycle,
                    "target": target,
                    "n_test": len(real_predictions),
                    **paired_result,
                    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
                    "bootstrap_confidence": BOOTSTRAP_CONFIDENCE,
                }
            )

    summary = pd.DataFrame(summary_rows)
    paired_summary = pd.DataFrame(paired_rows)
    artifacts = pd.DataFrame(artifact_records)

    summary_path = (
        OUTPUT_DIR
        / "single_model_mae_bootstrap_summary.csv"
    )
    paired_path = (
        OUTPUT_DIR
        / "single_model_paired_delta_mae_bootstrap.csv"
    )
    artifact_path = OUTPUT_DIR / "single_model_artifact_map.csv"

    summary.to_csv(
        summary_path,
        index=False,
        encoding="utf-8-sig",
    )
    paired_summary.to_csv(
        paired_path,
        index=False,
        encoding="utf-8-sig",
    )
    artifacts.to_csv(
        artifact_path,
        index=False,
        encoding="utf-8-sig",
    )

    figure_png = (
        OUTPUT_DIR
        / "fig2_single_model_mae_bootstrap_ci.png"
    )
    figure_pdf = (
        OUTPUT_DIR
        / "fig2_single_model_mae_bootstrap_ci.pdf"
    )
    plot_mae_with_bootstrap(
        summary=summary,
        output_png=figure_png,
        output_pdf=figure_pdf,
    )

    protocol = {
        "estimator": "original final single DNN model",
        "point_estimate": (
            "MAE calculated from each already-selected model folder's saved "
            "test_predictions.csv; final_best_model.h5 is used only "
            "when the prediction file is absent."
        ),
        "individual_error_bar": (
            "Percentile bootstrap interval of the single-model "
            "test MAE obtained by resampling held-out test samples."
        ),
        "paired_comparison": (
            "Paired percentile bootstrap interval of "
            "Delta MAE = MAE_augmented - MAE_real-only, using "
            "identical resampled test indices for both branches."
        ),
        "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        "bootstrap_confidence": BOOTSTRAP_CONFIDENCE,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "take_abs_s_ane": TAKE_ABS_S_ANE,
        "selected_branch": SELECTED_BRANCH,
        "summary_csv": str(summary_path),
        "paired_csv": str(paired_path),
        "artifact_map_csv": str(artifact_path),
        "figure_png": str(figure_png),
        "figure_pdf": str(figure_pdf),
    }
    with open(
        OUTPUT_DIR / "single_model_bootstrap_protocol.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            protocol,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\n" + "=" * 88)
    print("SINGLE-MODEL BOOTSTRAP ANALYSIS COMPLETED")
    print("=" * 88)
    print("\n[MAE with individual bootstrap intervals]")
    print(summary.to_string(index=False))
    print("\n[Paired Delta MAE bootstrap]")
    print(paired_summary.to_string(index=False))
    print(f"\nSaved summary: {summary_path}")
    print(f"Saved paired comparison: {paired_path}")
    print(f"Saved figure PNG: {figure_png}")
    print(f"Saved figure PDF: {figure_pdf}")


if __name__ == "__main__":
    main()


[Cycle 1] Both branches contain test_predictions.csv; test.csv is not required.
Loaded existing predictions: C:\Users\jp02291\Documents\Codex\2026-08-13\referenced-chatgpt-conversation-this-is-an\work\latest_review_260814\code_updated\ane-active-learning\analysis\bootstrap\cycle1\artifacts_dnn_base\test_predictions.csv


Loaded existing predictions: C:\Users\jp02291\Documents\Codex\2026-08-13\referenced-chatgpt-conversation-this-is-an\work\latest_review_260814\code_updated\ane-active-learning\analysis\bootstrap\cycle1\artifacts_dnn_augmented\test_predictions.csv


[Cycle 2] Both branches contain test_predictions.csv; test.csv is not required.
Loaded existing predictions: C:\Users\jp02291\Documents\Codex\2026-08-13\referenced-chatgpt-conversation-this-is-an\work\latest_review_260814\code_updated\ane-active-learning\analysis\bootstrap\cycle2\artifacts_dnn_base\test_predictions.csv


Loaded existing predictions: C:\Users\jp02291\Documents\Codex\2026-08-13\referenced-chatgpt-conversation-this-is-an\work\latest_review_260814\code_updated\ane-active-learning\analysis\bootstrap\cycle2\artifacts_dnn_augmented\test_predictions.csv


[Cycle 3] Both branches contain test_predictions.csv; test.csv is not required.
Loaded existing predictions: C:\Users\jp02291\Documents\Codex\2026-08-13\referenced-chatgpt-conversation-this-is-an\work\latest_review_260814\code_updated\ane-active-learning\analysis\bootstrap\cycle3\artifacts_dnn_base\test_predictions.csv


Loaded existing predictions: C:\Users\jp02291\Documents\Codex\2026-08-13\referenced-chatgpt-conversation-this-is-an\work\latest_review_260814\code_updated\ane-active-learning\analysis\bootstrap\cycle3\artifacts_dnn_augmented\test_predictions.csv



SINGLE-MODEL BOOTSTRAP ANALYSIS COMPLETED

[MAE with individual bootstrap intervals]
 cycle    branch target  n_test      MAE  MAE_CI_low  MAE_CI_high  MAE_error_low  MAE_error_high  bootstrap_iterations  bootstrap_confidence  selected_for_active_learning
     1 real-only    kxx       9 3.740193    2.419026     5.474198       1.321168        1.734005                  5000                  0.95                         False
     1 real-only  S_ANE       9 0.549156    0.337830     0.783433       0.211326        0.234277                  5000                  0.95                         False
     1 augmented    kxx       9 3.011283    1.919742     4.261151       1.091541        1.249868                  5000                  0.95                          True
     1 augmented  S_ANE       9 0.462655    0.241758     0.722897       0.220897        0.260242                  5000                  0.95                          True
     2 real-only    kxx      10 2.136695    1.355382     2.